<a href="https://colab.research.google.com/github/utpalssg/pythonai/blob/newBranch/TensorTry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from getpass import getpass
token = getpass('Enter your GitHub token:')
!git clone --recurse-submodules https://{token}@github.com/utpalssg/pythonai.git

Enter your GitHub token:··········
Cloning into 'pythonai'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 78 (delta 36), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 1.84 MiB | 4.69 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [2]:
import torch
from torchvision import models, transforms

# using cpu
#model = models.resnet50(pretrained=True)

# using gpu
model = models.resnet50(pretrained=True).to("cuda")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 150MB/s]


In [3]:
from PIL import Image

img = Image.open("/content/pythonai/inference/threeofus.jpg")


In [4]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# Apply the transform to the original PIL image from cell cOmxknVSJv8t
img_tensor = transform(img)
img_tensor.shape

torch.Size([3, 224, 224])

In [5]:
import torch

img_batch = torch.unsqueeze(img_tensor, 0).to("cuda")
img_batch.shape

torch.Size([1, 3, 224, 224])

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)             # Move model to GPU
img_batch = img_batch.to(device)     # Move input to GPU

model.eval()
with torch.no_grad():
    outputs = model(img_batch)
prob = torch.nn.functional.softmax(outputs[0], dim=0)

In [10]:
import pandas as pd

labels = pd.read_csv("https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt", header=None)
labels[0][3]

'tiger shark'

In [16]:
topk=5
prob, class_number = torch.topk(probs, topk)
for i in range(topk):
    probability = prob[i].item()
    class_name = labels[0][int(class_number[i])]
    print(f"{class_name}: {probability * 100:.2f}%")

military uniform: 38.19%
knee pad: 9.51%
tricycle: 6.58%
swimming trunks: 6.07%
toyshop: 5.56%
